In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/ladydisdain5911/python_course_repo/refs/heads/main/train.csv')

Основная метрика: F1-score - так как задача несбалансированная (выжило ~38%, погибло ~62%)
и нам важно учитывать как precision (точность предсказания выживших),
так и recall (полноту выявления выживших).

Дополнительные метрики: ROC-AUC - для оценки общей производительности модели,
и Accuracy - как базовый показатель.

In [3]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = df[features]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Размер тренировочной выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")
print(f"Доля выживших в тренировочной выборке: {y_train.mean():.3f}")
print(f"Доля выживших в тестовой выборке: {y_test.mean():.3f}")

Размер тренировочной выборки: (712, 7)
Размер тестовой выборки: (179, 7)
Доля выживших в тренировочной выборке: 0.383
Доля выживших в тестовой выборке: 0.385


In [4]:
numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['Pclass', 'Sex', 'Embarked']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [5]:
# Константное предсказание (все пассажиры погибли - наиболее частотный класс)
baseline_pred = np.zeros(len(y_test))

# Вычисляем метрики для бейзлайна
baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_precision = precision_score(y_test, baseline_pred, zero_division=0)
baseline_recall = recall_score(y_test, baseline_pred, zero_division=0)
baseline_f1 = f1_score(y_test, baseline_pred, zero_division=0)

print("=== РЕЗУЛЬТАТЫ БЕЙЗЛАЙН МОДЕЛИ ===")
print(f"Accuracy: {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall: {baseline_recall:.4f}")
print(f"F1-score: {baseline_f1:.4f}")

=== РЕЗУЛЬТАТЫ БЕЙЗЛАЙН МОДЕЛИ ===
Accuracy: 0.6145
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000


In [6]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

ml_accuracy = accuracy_score(y_test, y_pred)
ml_precision = precision_score(y_test, y_pred)
ml_recall = recall_score(y_test, y_pred)
ml_f1 = f1_score(y_test, y_pred)
ml_roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=== РЕЗУЛЬТАТЫ ML МОДЕЛИ ===")
print(f"Accuracy: {ml_accuracy:.4f}")
print(f"Precision: {ml_precision:.4f}")
print(f"Recall: {ml_recall:.4f}")
print(f"F1-score: {ml_f1:.4f}")
print(f"ROC-AUC: {ml_roc_auc:.4f}")

=== РЕЗУЛЬТАТЫ ML МОДЕЛИ ===
Accuracy: 0.7989
Precision: 0.7797
Recall: 0.6667
F1-score: 0.7188
ROC-AUC: 0.8436


In [7]:
comparison = pd.DataFrame({
    'Метрика': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC'],
    'Бейзлайн': [baseline_accuracy, baseline_precision, baseline_recall, baseline_f1, '-'],
    'ML-модель': [ml_accuracy, ml_precision, ml_recall, ml_f1, ml_roc_auc]
})

print("=== СРАВНИТЕЛЬНЫЙ АНАЛИЗ ===")
print(comparison)

print("\n=== ДЕТАЛЬНЫЙ ОТЧЕТ ML МОДЕЛИ ===")
print(classification_report(y_test, y_pred, target_names=['Погиб (0)', 'Выжил (1)']))

=== СРАВНИТЕЛЬНЫЙ АНАЛИЗ ===
     Метрика  Бейзлайн  ML-модель
0   Accuracy  0.614525   0.798883
1  Precision       0.0   0.779661
2     Recall       0.0   0.666667
3   F1-score       0.0   0.718750
4    ROC-AUC         -   0.843610

=== ДЕТАЛЬНЫЙ ОТЧЕТ ML МОДЕЛИ ===
              precision    recall  f1-score   support

   Погиб (0)       0.81      0.88      0.84       110
   Выжил (1)       0.78      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.79      0.77      0.78       179
weighted avg       0.80      0.80      0.80       179



In [8]:
feature_names = (numeric_features +
                 list(model.named_steps['preprocessor']
                     .named_transformers_['cat']
                     .named_steps['onehot']
                     .get_feature_names_out(categorical_features)))

coefficients = model.named_steps['classifier'].coef_[0]

feature_importance = pd.DataFrame({
    'Признак': feature_names,
    'Важность': np.abs(coefficients)
}).sort_values('Важность', ascending=False)

print("=== ВАЖНОСТЬ ПРИЗНАКОВ ===")
print(feature_importance)

=== ВАЖНОСТЬ ПРИЗНАКОВ ===
      Признак  Важность
6    Sex_male  2.529833
5    Pclass_3  1.919663
4    Pclass_2  0.734868
0         Age  0.468895
8  Embarked_S  0.438453
1       SibSp  0.260728
7  Embarked_Q  0.228675
3        Fare  0.174708
2       Parch  0.069937
